In [1]:
## INGESTION
## SPLIT VALID INTO VAL AND TEST

from pathlib import Path
import shutil
import random

# ==========================
# CONFIG
# ==========================
DATASET_ROOT = Path("/workspace/data/data_set/tomato-flower")
SOURCE_DIR = DATASET_ROOT / "valid"
VAL_RATIO = 0.5
RANDOM_SEED = 42

# ==========================
# PATHS
# ==========================
source_images = SOURCE_DIR / "images"
source_labels = SOURCE_DIR / "labels"

val_dir = DATASET_ROOT / "val"
test_dir = DATASET_ROOT / "test"

val_images = val_dir / "images"
val_labels = val_dir / "labels"
test_images = test_dir / "images"
test_labels = test_dir / "labels"

# Create output directories
for d in [val_images, val_labels, test_images, test_labels]:
    d.mkdir(parents=True, exist_ok=True)

# ==========================
# COLLECT IMAGE FILES
# ==========================
valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_files = sorted([f for f in source_images.iterdir() if f.is_file() and f.suffix.lower() in valid_exts])

if not image_files:
    raise ValueError(f"No image files found in {source_images}")

# ==========================
# SHUFFLE + SPLIT
# ==========================
random.seed(RANDOM_SEED)
random.shuffle(image_files)

split_idx = int(len(image_files) * VAL_RATIO)
val_files = image_files[:split_idx]
test_files = image_files[split_idx:]

print(f"Total source images: {len(image_files)}")
print(f"Validation images:   {len(val_files)}")
print(f"Test images:         {len(test_files)}")

# ==========================
# MOVE IMAGE + LABEL PAIR
# ==========================
def move_pair(img_path: Path, src_label_dir: Path, dest_img_dir: Path, dest_label_dir: Path):
    label_path = src_label_dir / f"{img_path.stem}.txt"

    # Move image
    shutil.copy2(str(img_path), str(dest_img_dir / img_path.name))

    # Move matching label
    if label_path.exists():
        shutil.copy2(str(label_path), str(dest_label_dir / label_path.name))
    else:
        print(f"WARNING! Missing label for image: {img_path.name}")

# Move val split
for img in val_files:
    move_pair(img, source_labels, val_images, val_labels)

# Move test split
for img in test_files:
    move_pair(img, source_labels, test_images, test_labels)

print("DONE! Split complete.")

Total source images: 3636
Validation images:   1818
Test images:         1818
DONE! Split complete.


In [2]:
## VERIFY INGESTION
## COUNT AMOUNT OF IMAGES IN EACH FOLDER

from pathlib import Path

DATASET_ROOT = Path("/workspace/data/data_set/tomato-flower")

for split in ["train", "val", "test"]:
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"

    img_count = len([p for p in img_dir.iterdir() if p.is_file()]) if img_dir.exists() else 0
    lbl_count = len([p for p in lbl_dir.iterdir() if p.is_file()]) if lbl_dir.exists() else 0

    print(f"{split.upper()}:")
    print(f"  images = {img_count}")
    print(f"  labels = {lbl_count}")
    print()

TRAIN:
  images = 14545
  labels = 14545

VAL:
  images = 1818
  labels = 1818

TEST:
  images = 1818
  labels = 1818



In [3]:
## DATA YAML UPDATER
## CREATES THE NEW DATA CONFIG FILE FOR THE NEW FOLDERS

from pathlib import Path
import yaml

data_yaml_path = Path("/workspace/flower-classifier/config/data.yaml")
data_yaml_path.parent.mkdir(parents=True, exist_ok=True)

data_config = {
    "train": "/workspace/data/data_set/tomato-flower/train/images",
    "val": "/workspace/data/data_set/tomato-flower/val/images",
    "test": "/workspace/data/data_set/tomato-flower/test/images",
    "nc": 3,
    "names": ["Bud", "Anthesis", "Post-Anthesis"],
}

with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print(f"DONE! Wrote data.yaml to: {data_yaml_path}")
print(data_yaml_path.read_text())

DONE! Wrote data.yaml to: /workspace/flower-classifier/config/data.yaml
train: /workspace/data/data_set/tomato-flower/train/images
val: /workspace/data/data_set/tomato-flower/val/images
test: /workspace/data/data_set/tomato-flower/test/images
nc: 3
names:
- Bud
- Anthesis
- Post-Anthesis

